In [1]:
!pip install pandas numpy matplotlib pyreadstat scikit-learn statsmodels xlsxwriter


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 12.8 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# ============================================================
# 0. SETTINGS
# ============================================================

DATA_PATH = Path("kids_pooled_village_treat.dta")

ROOT_OUTPUT = Path("final_project_outputs")
DID_OUTPUT = ROOT_OUTPUT / "did_outputs"
ML_OUTPUT = ROOT_OUTPUT / "ml_outputs"
REPORT_OUTPUT = ROOT_OUTPUT / "report_ready_outputs"

DID_OUTPUT.mkdir(parents=True, exist_ok=True)
ML_OUTPUT.mkdir(parents=True, exist_ok=True)
REPORT_OUTPUT.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TEST_SIZE = 0.30
np.random.seed(RANDOM_SEED)

# ============================================================
# 1. LOAD AND CLEAN DATA
# ============================================================

df = pd.read_stata(DATA_PATH, convert_categoricals=False)

numeric_cols = [
    "year", "IDPSU", "STATEID", "treat_v", "post", "girl",
    "age", "age2", "ontrack", "COPC", "HHED_ADULT",
    "gradegap", "grade", "expected_grade"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "age2" not in df.columns or df["age2"].isna().all():
    df["age2"] = df["age"] ** 2

if "expected_grade" not in df.columns or df["expected_grade"].isna().all():
    df["expected_grade"] = df["age"] - 5

if ("ontrack" not in df.columns or df["ontrack"].isna().all()) and "grade" in df.columns:
    df["ontrack"] = (df["grade"] >= df["expected_grade"]).astype(float)

if ("gradegap" not in df.columns or df["gradegap"].isna().all()) and "grade" in df.columns:
    # gradegap = actual grade - expected grade
    df["gradegap"] = df["grade"] - df["expected_grade"]

if "post" not in df.columns or df["post"].isna().all():
    if "year" in df.columns:
        max_year = df["year"].dropna().max()
        df["post"] = (df["year"] == max_year).astype(float)
    else:
        raise ValueError("Need either post or year variable.")

df["STATEID"] = df["STATEID"].astype("Int64").astype(str)

if "year" in df.columns and df["year"].notna().any():
    years_sorted = sorted(df["year"].dropna().unique().tolist())
    pre_year = int(years_sorted[0])
    post_year = int(years_sorted[-1])
else:
    pre_year = 0
    post_year = 1

# Severe delay target
if "grade" in df.columns and "expected_grade" in df.columns:
    df["years_behind"] = df["expected_grade"] - df["grade"]
elif "gradegap" in df.columns:
    df["years_behind"] = -df["gradegap"]
else:
    raise ValueError("Need either grade and expected_grade, or gradegap.")

df["severe_offtrack"] = (df["years_behind"] >= 2).astype(int)

# ============================================================
# 2. HELPERS
# ============================================================

def save_text(path, text):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def regression_table(model, keep_terms=None):
    out = pd.DataFrame({
        "term": model.params.index,
        "coef": model.params.values,
        "std_err": model.bse.values,
        "t_stat": model.tvalues.values,
        "p_value": model.pvalues.values
    })
    if keep_terms is not None:
        out = out[out["term"].isin(keep_terms)].copy()
    return out.reset_index(drop=True)

def run_clustered_ols(formula, data, cluster_col):
    return smf.ols(formula=formula, data=data).fit(
        cov_type="cluster",
        cov_kwds={"groups": data[cluster_col]}
    )

def make_balance_table(data, group_col, vars_list):
    rows = []
    for v in vars_list:
        temp = (
            data.groupby(group_col)[v]
            .agg(mean="mean", std="std", n="count")
            .reset_index()
        )
        temp["variable"] = v
        rows.append(temp)
    out = pd.concat(rows, ignore_index=True)
    return out[["variable", group_col, "mean", "std", "n"]]

def subgroup_summary_tidy(df_in, group_col):
    out = (
        df_in.groupby(group_col)
        .agg(
            n=("actual_severe_offtrack", "size"),
            actual_severe_rate=("actual_severe_offtrack", "mean"),
            mean_predicted_severe_prob=("predicted_prob_severe_offtrack", "mean"),
            median_predicted_severe_prob=("predicted_prob_severe_offtrack", "median"),
            mean_years_behind=("years_behind", "mean"),
            median_years_behind=("years_behind", "median")
        )
        .reset_index()
        .rename(columns={group_col: "group_value"})
    )
    out["grouping_variable"] = group_col
    return out[
        [
            "grouping_variable", "group_value", "n",
            "actual_severe_rate",
            "mean_predicted_severe_prob", "median_predicted_severe_prob",
            "mean_years_behind", "median_years_behind"
        ]
    ]

def make_raw_means_table(raw_df, outcome_col, pre_label, post_label):
    # raw_df expected columns: treat_v, post, outcome_col
    wide = raw_df.pivot(index="treat_v", columns="post", values=outcome_col).reset_index()
    wide = wide.rename(columns={
        0.0: pre_label,
        1.0: post_label,
        0: pre_label,
        1: post_label,
        "treat_v": "Group"
    })
    wide["Group"] = wide["Group"].map({
        0.0: "Control Villages (treat_v = 0)",
        1.0: "Treated Villages (treat_v = 1)",
        0: "Control Villages (treat_v = 0)",
        1: "Treated Villages (treat_v = 1)"
    })
    return wide

def make_balance_table_report(balance_df):
    # Converts long-format balance table to a compact wide report table
    report = balance_df.pivot(index="variable", columns="treat_v", values="mean").reset_index()
    report = report.rename(columns={
        "variable": "Variable",
        0.0: "Control Villages (treat_v = 0)",
        1.0: "Treated Villages (treat_v = 1)",
        0: "Control Villages (treat_v = 0)",
        1: "Treated Villages (treat_v = 1)"
    })
    return report

def autosize_excel(writer, sheet_name, df_to_size):
    worksheet = writer.sheets[sheet_name]
    for i, col in enumerate(df_to_size.columns):
        max_len = max(
            len(str(col)),
            df_to_size[col].astype(str).map(len).max() if len(df_to_size) > 0 else 0
        )
        worksheet.set_column(i, i, min(max_len + 2, 40))

# ============================================================
# 3. SECTION A: DID DESCRIPTIVES
# ============================================================

print("=" * 70)
print("SECTION A: DIFFERENCE-IN-DIFFERENCES")
print("=" * 70)

balance_vars = ["age", "girl", "ontrack", "COPC", "HHED_ADULT"]

if "year" in df.columns and df["year"].notna().any():
    balance_pre = make_balance_table(
        data=df.loc[df["year"] == pre_year].dropna(subset=["treat_v"]).copy(),
        group_col="treat_v",
        vars_list=balance_vars
    )
    balance_post = make_balance_table(
        data=df.loc[df["year"] == post_year].dropna(subset=["treat_v"]).copy(),
        group_col="treat_v",
        vars_list=balance_vars
    )

    balance_pre.to_csv(DID_OUTPUT / f"balance_table_{pre_year}.csv", index=False)
    balance_post.to_csv(DID_OUTPUT / f"balance_table_{post_year}.csv", index=False)

raw_means_ontrack = (
    df.groupby(["treat_v", "post"], dropna=False)["ontrack"]
      .mean()
      .reset_index()
      .sort_values(["treat_v", "post"])
)
raw_means_ontrack.to_csv(DID_OUTPUT / "raw_means_ontrack.csv", index=False)

raw_means_severe = (
    df.groupby(["treat_v", "post"], dropna=False)["severe_offtrack"]
      .mean()
      .reset_index()
      .sort_values(["treat_v", "post"])
)
raw_means_severe.to_csv(DID_OUTPUT / "raw_means_severe_offtrack.csv", index=False)

print("Saved descriptive tables.")
print()

# ============================================================
# 4. SECTION A: BENCHMARK DID + GRADEGAP ROBUSTNESS
# ============================================================

did_vars = [
    "ontrack", "treat_v", "post", "girl",
    "age", "age2", "COPC", "HHED_ADULT",
    "STATEID", "IDPSU"
]
did_df = df[did_vars].dropna().copy()

formula_did = (
    "ontrack ~ treat_v * post + girl + age + age2 + COPC + HHED_ADULT + C(STATEID)"
)
m_did = run_clustered_ols(formula_did, did_df, "IDPSU")

gradegap_vars = [
    "gradegap", "treat_v", "post", "girl",
    "age", "age2", "COPC", "HHED_ADULT",
    "STATEID", "IDPSU"
]
gradegap_df = df[gradegap_vars].dropna().copy()

formula_gradegap = (
    "gradegap ~ treat_v * post + girl + age + age2 + COPC + HHED_ADULT + C(STATEID)"
)
m_gradegap = run_clustered_ols(formula_gradegap, gradegap_df, "IDPSU")

reg_did = regression_table(
    m_did,
    keep_terms=[
        "Intercept", "treat_v", "post", "treat_v:post",
        "girl", "age", "age2", "COPC", "HHED_ADULT"
    ]
)
reg_gradegap = regression_table(
    m_gradegap,
    keep_terms=[
        "Intercept", "treat_v", "post", "treat_v:post",
        "girl", "age", "age2", "COPC", "HHED_ADULT"
    ]
)

reg_did.to_csv(DID_OUTPUT / "regression_did_ontrack.csv", index=False)
reg_gradegap.to_csv(DID_OUTPUT / "regression_did_gradegap.csv", index=False)

save_text(DID_OUTPUT / "summary_did_ontrack.txt", m_did.summary().as_text())
save_text(DID_OUTPUT / "summary_did_gradegap.txt", m_gradegap.summary().as_text())

did_main_summary = pd.DataFrame({
    "model": ["Benchmark DiD: ontrack", "Robustness DiD: gradegap"],
    "treat_post_coef": [
        m_did.params.get("treat_v:post", np.nan),
        m_gradegap.params.get("treat_v:post", np.nan)
    ],
    "treat_post_se": [
        m_did.bse.get("treat_v:post", np.nan),
        m_gradegap.bse.get("treat_v:post", np.nan)
    ],
    "n_obs": [int(m_did.nobs), int(m_gradegap.nobs)],
    "r_squared": [m_did.rsquared, m_gradegap.rsquared]
})
did_main_summary.to_csv(DID_OUTPUT / "did_main_summary.csv", index=False)

print("Saved benchmark DiD outputs.")
print(did_main_summary)
print()

# ============================================================
# 5. SECTION B: MACHINE LEARNING DATA
# ============================================================

print("=" * 70)
print("SECTION B: MACHINE LEARNING FOR SEVERE EDUCATIONAL DELAY")
print("=" * 70)

ml_use_cols = [
    "severe_offtrack", "years_behind",
    "girl", "age", "age2", "COPC", "HHED_ADULT",
    "treat_v", "post", "STATEID"
]
ml_data = df[ml_use_cols].dropna().copy()

ml_target_summary = pd.DataFrame({
    "metric": ["n_obs", "severe_offtrack_rate", "mean_years_behind", "median_years_behind"],
    "value": [
        len(ml_data),
        ml_data["severe_offtrack"].mean(),
        ml_data["years_behind"].mean(),
        ml_data["years_behind"].median()
    ]
})
ml_target_summary.to_csv(ML_OUTPUT / "ml_target_summary.csv", index=False)

print(ml_target_summary)
print()

# ============================================================
# 6. SECTION B: TRAIN / TEST SPLIT
# ============================================================

X = ml_data.drop(columns=["severe_offtrack", "years_behind"])
y = ml_data["severe_offtrack"]
yb = ml_data["years_behind"]

X_train, X_test, y_train, y_test, yb_train, yb_test = train_test_split(
    X, y, yb,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y
)

numeric_features = ["girl", "age", "age2", "COPC", "HHED_ADULT", "treat_v", "post"]
categorical_features = ["STATEID"]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# ============================================================
# 7. SECTION B: MODELS + TUNING
# ============================================================

pipe_logit = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))
    ]
)
grid_logit = {"model__C": [0.01, 0.1, 1, 5]}

pipe_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1))
    ]
)
grid_rf = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [6, 12, None],
    "model__min_samples_leaf": [5, 20]
}

searches = {
    "Logistic Regression": GridSearchCV(
        pipe_logit, grid_logit, cv=cv, scoring="average_precision", n_jobs=-1
    ),
    "Random Forest": GridSearchCV(
        pipe_rf, grid_rf, cv=cv, scoring="average_precision", n_jobs=-1
    )
}

# ============================================================
# 8. SECTION B: FIT MODELS
# ============================================================

fitted_models = {}
metrics_rows = []

for name, search in searches.items():
    print(f"Running ML model: {name}")
    search.fit(X_train, y_train)
    best_estimator = search.best_estimator_
    fitted_models[name] = best_estimator

    y_pred = best_estimator.predict(X_test)
    y_prob = best_estimator.predict_proba(X_test)[:, 1]

    metrics_rows.append({
        "model": name,
        "best_cv_avg_precision": search.best_score_,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_roc_auc": roc_auc_score(y_test, y_prob),
        "test_avg_precision": average_precision_score(y_test, y_prob),
        "test_precision": precision_score(y_test, y_pred, zero_division=0),
        "test_recall": recall_score(y_test, y_pred, zero_division=0),
        "test_f1": f1_score(y_test, y_pred, zero_division=0),
        "best_params": str(search.best_params_)
    })

model_comparison_metrics = pd.DataFrame(metrics_rows).sort_values(
    "best_cv_avg_precision", ascending=False
)
model_comparison_metrics.to_csv(ML_OUTPUT / "model_comparison_metrics_severe.csv", index=False)

best_model_name = model_comparison_metrics.iloc[0]["model"]
best_model = fitted_models[best_model_name]

print()
print(model_comparison_metrics)
print()
print(f"Selected best model: {best_model_name}")
print()

# ============================================================
# 9. SECTION B: SAVE TEST PREDICTIONS
# ============================================================

test_results = X_test.copy()
test_results["actual_severe_offtrack"] = y_test.values
test_results["years_behind"] = yb_test.values
test_results["predicted_prob_severe_offtrack"] = best_model.predict_proba(X_test)[:, 1]
test_results["predicted_class_severe_offtrack"] = best_model.predict(X_test)

test_results["severe_risk_group"] = pd.qcut(
    test_results["predicted_prob_severe_offtrack"].rank(method="first"),
    q=5,
    labels=["Lowest risk", "Low risk", "Medium risk", "High risk", "Highest risk"]
)

test_results.to_csv(ML_OUTPUT / "best_model_test_predictions_severe.csv", index=False)

# ============================================================
# 10. SECTION B: ROC / PR / CONFUSION MATRIX
# ============================================================

plt.figure(figsize=(7, 6))
RocCurveDisplay.from_estimator(best_model, X_test, y_test)
plt.title(f"ROC Curve - {best_model_name}")
plt.tight_layout()
plt.savefig(ML_OUTPUT / "plot_roc_curve_best_model_severe.png", dpi=300)
plt.close()

plt.figure(figsize=(7, 6))
PrecisionRecallDisplay.from_estimator(best_model, X_test, y_test)
plt.title(f"Precision-Recall Curve - {best_model_name}")
plt.tight_layout()
plt.savefig(ML_OUTPUT / "plot_pr_curve_best_model_severe.png", dpi=300)
plt.close()

cm = confusion_matrix(y_test, best_model.predict(X_test))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.tight_layout()
plt.savefig(ML_OUTPUT / "plot_confusion_matrix_best_model_severe.png", dpi=300)
plt.close()

# ============================================================
# 11. SECTION B: FEATURE IMPORTANCE
# ============================================================

model_obj = best_model.named_steps["model"]
transformed_feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()

if hasattr(model_obj, "feature_importances_"):
    feature_importance_builtin = pd.DataFrame({
        "feature": transformed_feature_names,
        "importance": model_obj.feature_importances_
    }).sort_values("importance", ascending=False)
    feature_importance_builtin.to_csv(ML_OUTPUT / "feature_importance_builtin_severe.csv", index=False)

    top_builtin = feature_importance_builtin.head(15).sort_values("importance", ascending=True)
    plt.figure(figsize=(8, 6))
    plt.barh(top_builtin["feature"], top_builtin["importance"])
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.title(f"Built-in Feature Importance - {best_model_name}")
    plt.tight_layout()
    plt.savefig(ML_OUTPUT / "plot_feature_importance_builtin_severe.png", dpi=300)
    plt.close()

perm = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=10,
    random_state=RANDOM_SEED,
    scoring="average_precision",
    n_jobs=-1
)

feature_importance_permutation = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

feature_importance_permutation.to_csv(
    ML_OUTPUT / "feature_importance_permutation_severe.csv", index=False
)

top_perm = feature_importance_permutation.head(10).sort_values("importance_mean", ascending=True)
plt.figure(figsize=(8, 6))
plt.barh(top_perm["feature"], top_perm["importance_mean"])
plt.xlabel("Permutation importance (Average Precision loss)")
plt.ylabel("Feature")
plt.title(f"Permutation Importance - {best_model_name}")
plt.tight_layout()
plt.savefig(ML_OUTPUT / "plot_feature_importance_permutation_severe.png", dpi=300)
plt.close()

# ============================================================
# 12. SECTION B: SUBGROUP SUMMARIES
# ============================================================

copc_med = test_results["COPC"].median()
edu_med = test_results["HHED_ADULT"].median()

test_results["consumption_group"] = np.where(
    test_results["COPC"] <= copc_med, "Lower consumption", "Higher consumption"
)
test_results["education_group"] = np.where(
    test_results["HHED_ADULT"] <= edu_med, "Lower adult education", "Higher adult education"
)
test_results["gender_group"] = np.where(
    test_results["girl"] == 1, "Girls", "Boys"
)
test_results["age_group"] = np.where(
    test_results["age"] <= 10, "Age 6-10", "Age 11-14"
)
test_results["village_group"] = np.where(
    test_results["treat_v"] == 1, "MGNREGA village", "Non-MGNREGA village"
)

sub_gender = subgroup_summary_tidy(test_results, "gender_group")
sub_age = subgroup_summary_tidy(test_results, "age_group")
sub_cons = subgroup_summary_tidy(test_results, "consumption_group")
sub_edu = subgroup_summary_tidy(test_results, "education_group")
sub_village = subgroup_summary_tidy(test_results, "village_group")

subgroup_severe_risk_summary = pd.concat(
    [sub_gender, sub_age, sub_cons, sub_edu, sub_village],
    ignore_index=True
)
subgroup_severe_risk_summary.to_csv(
    ML_OUTPUT / "subgroup_severe_risk_summary.csv", index=False
)

# ============================================================
# 13. SECTION B: RISK PLOTS
# ============================================================

for group_col, title, filename in [
    ("gender_group", "Predicted severe off-track risk by gender", "plot_severe_risk_by_gender.png"),
    ("age_group", "Predicted severe off-track risk by age group", "plot_severe_risk_by_age.png"),
    ("consumption_group", "Predicted severe off-track risk by consumption group", "plot_severe_risk_by_consumption.png"),
    ("education_group", "Predicted severe off-track risk by adult education group", "plot_severe_risk_by_education.png"),
    ("village_group", "Predicted severe off-track risk by village type", "plot_severe_risk_by_village_type.png"),
]:
    temp = (
        test_results.groupby(group_col)["predicted_prob_severe_offtrack"]
        .mean()
        .reset_index()
    )
    plt.figure(figsize=(7, 5))
    plt.bar(temp[group_col], temp["predicted_prob_severe_offtrack"])
    plt.ylabel("Mean predicted severe off-track risk")
    plt.xlabel("")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(ML_OUTPUT / filename, dpi=300)
    plt.close()

plt.figure(figsize=(8, 6))
plt.hist(test_results["predicted_prob_severe_offtrack"], bins=40)
plt.xlabel("Predicted probability of being severely off-track")
plt.ylabel("Frequency")
plt.title(f"Distribution of predicted severe off-track risk - {best_model_name}")
plt.tight_layout()
plt.savefig(ML_OUTPUT / "plot_severe_risk_distribution.png", dpi=300)
plt.close()

# ============================================================
# 14. SECTION B: HIGH-RISK PROFILE
# ============================================================

high_risk = test_results[test_results["severe_risk_group"] == "Highest risk"].copy()

high_risk_profile = pd.DataFrame({
    "metric": [
        "n_highest_risk",
        "share_girls",
        "mean_age",
        "median_age",
        "mean_COPC",
        "median_COPC",
        "mean_HHED_ADULT",
        "median_HHED_ADULT",
        "share_MGNREGA_village",
        "actual_severe_rate",
        "mean_years_behind",
        "median_years_behind",
        "mean_predicted_severe_prob",
        "median_predicted_severe_prob"
    ],
    "value": [
        len(high_risk),
        high_risk["girl"].mean(),
        high_risk["age"].mean(),
        high_risk["age"].median(),
        high_risk["COPC"].mean(),
        high_risk["COPC"].median(),
        high_risk["HHED_ADULT"].mean(),
        high_risk["HHED_ADULT"].median(),
        high_risk["treat_v"].mean(),
        high_risk["actual_severe_offtrack"].mean(),
        high_risk["years_behind"].mean(),
        high_risk["years_behind"].median(),
        high_risk["predicted_prob_severe_offtrack"].mean(),
        high_risk["predicted_prob_severe_offtrack"].median()
    ]
})
high_risk_profile.to_csv(ML_OUTPUT / "high_risk_profile_severe.csv", index=False)

# ============================================================
# 15. CREATE REPORT-READY TABLES (EXCEL)
# ============================================================

# Raw means report tables
report_raw_means_ontrack = make_raw_means_table(
    raw_means_ontrack, "ontrack",
    f"{pre_year} (post=0)", f"{post_year} (post=1)"
)

report_raw_means_severe = make_raw_means_table(
    raw_means_severe, "severe_offtrack",
    f"{pre_year} (post=0)", f"{post_year} (post=1)"
)

# Balance table report
if "year" in df.columns and df["year"].notna().any():
    report_balance_pre = make_balance_table_report(balance_pre)
else:
    report_balance_pre = pd.DataFrame()

# Report-ready benchmark DiD table
report_did_table = pd.DataFrame({
    "Model": ["On track", "Grade gap"],
    "MGNREGA village (treat_v)": [
        m_did.params.get("treat_v", np.nan),
        m_gradegap.params.get("treat_v", np.nan)
    ],
    "post": [
        m_did.params.get("post", np.nan),
        m_gradegap.params.get("post", np.nan)
    ],
    "treat × post": [
        m_did.params.get("treat_v:post", np.nan),
        m_gradegap.params.get("treat_v:post", np.nan)
    ],
    "girl": [
        m_did.params.get("girl", np.nan),
        m_gradegap.params.get("girl", np.nan)
    ],
    "Observations": [int(m_did.nobs), int(m_gradegap.nobs)],
    "R2": [m_did.rsquared, m_gradegap.rsquared]
})

# ML model comparison report
report_ml_model_table = model_comparison_metrics.copy()

# Feature importance report
report_feature_importance = feature_importance_permutation.copy()

# Subgroup summary report
report_subgroup_summary = subgroup_severe_risk_summary.copy()

# High-risk profile report
report_high_risk_profile = high_risk_profile.copy()

report_workbook_path = REPORT_OUTPUT / "report_ready_tables.xlsx"

with pd.ExcelWriter(report_workbook_path, engine="xlsxwriter") as writer:
    report_raw_means_ontrack.to_excel(writer, sheet_name="RawMeans_OnTrack", index=False)
    autosize_excel(writer, "RawMeans_OnTrack", report_raw_means_ontrack)

    report_raw_means_severe.to_excel(writer, sheet_name="RawMeans_SevereDelay", index=False)
    autosize_excel(writer, "RawMeans_SevereDelay", report_raw_means_severe)

    if not report_balance_pre.empty:
        report_balance_pre.to_excel(writer, sheet_name=f"Balance_{pre_year}", index=False)
        autosize_excel(writer, f"Balance_{pre_year}", report_balance_pre)

    report_did_table.to_excel(writer, sheet_name="DiD_Report_Table", index=False)
    autosize_excel(writer, "DiD_Report_Table", report_did_table)

    report_ml_model_table.to_excel(writer, sheet_name="ML_ModelComparison", index=False)
    autosize_excel(writer, "ML_ModelComparison", report_ml_model_table)

    report_feature_importance.to_excel(writer, sheet_name="ML_FeatureImportance", index=False)
    autosize_excel(writer, "ML_FeatureImportance", report_feature_importance)

    report_subgroup_summary.to_excel(writer, sheet_name="ML_SubgroupSummary", index=False)
    autosize_excel(writer, "ML_SubgroupSummary", report_subgroup_summary)

    report_high_risk_profile.to_excel(writer, sheet_name="ML_HighRiskProfile", index=False)
    autosize_excel(writer, "ML_HighRiskProfile", report_high_risk_profile)

# ============================================================
# 16. README / OUTPUT GUIDE
# ============================================================

did_readme = f"""
DID OUTPUTS IN: {DID_OUTPUT.resolve()}

Files:
- balance_table_{pre_year}.csv
- balance_table_{post_year}.csv
- raw_means_ontrack.csv
- raw_means_severe_offtrack.csv
- regression_did_ontrack.csv
- regression_did_gradegap.csv
- summary_did_ontrack.txt
- summary_did_gradegap.txt
- did_main_summary.csv
"""
save_text(DID_OUTPUT / "README_did_outputs.txt", did_readme)

ml_readme = f"""
ML OUTPUTS IN: {ML_OUTPUT.resolve()}

Core tables:
- ml_target_summary.csv
- model_comparison_metrics_severe.csv
- best_model_test_predictions_severe.csv
- feature_importance_permutation_severe.csv
- subgroup_severe_risk_summary.csv
- high_risk_profile_severe.csv

Plots:
- plot_roc_curve_best_model_severe.png
- plot_pr_curve_best_model_severe.png
- plot_confusion_matrix_best_model_severe.png
- plot_feature_importance_permutation_severe.png
- plot_severe_risk_distribution.png
- plot_severe_risk_by_gender.png
- plot_severe_risk_by_age.png
- plot_severe_risk_by_consumption.png
- plot_severe_risk_by_education.png
- plot_severe_risk_by_village_type.png
"""
save_text(ML_OUTPUT / "README_ml_outputs.txt", ml_readme)

report_readme = f"""
REPORT-READY TABLES IN: {REPORT_OUTPUT.resolve()}

Excel workbook:
- report_ready_tables.xlsx

Sheets:
- RawMeans_OnTrack
- RawMeans_SevereDelay
- Balance_{pre_year}
- DiD_Report_Table
- ML_ModelComparison
- ML_FeatureImportance
- ML_SubgroupSummary
- ML_HighRiskProfile
"""
save_text(REPORT_OUTPUT / "README_report_ready_tables.txt", report_readme)

# ============================================================
# 17. FINAL SUMMARY
# ============================================================

print("=" * 70)
print("ALL PROJECT OUTPUTS CREATED")
print("=" * 70)
print("DID output folder:", DID_OUTPUT.resolve())
print("ML output folder:", ML_OUTPUT.resolve())
print("Report-ready tables folder:", REPORT_OUTPUT.resolve())
print("Report-ready workbook:", report_workbook_path.resolve())
print()
print("Key DID summary:")
print(did_main_summary)
print()
print("Key ML model comparison:")
print(model_comparison_metrics)
print()
print("Done.")

SECTION A: DIFFERENCE-IN-DIFFERENCES
Saved descriptive tables.



/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 39, but rank is 38
  warnings.warn('covariance of constraints does not have full '
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 39, but rank is 38
  warnings.warn('covariance of constraints does not have full '


Saved benchmark DiD outputs.
                      model  treat_post_coef  treat_post_se  n_obs  r_squared
0    Benchmark DiD: ontrack         0.062825       0.013924  54058   0.099973
1  Robustness DiD: gradegap         0.251252       0.050204  54058   0.250823

SECTION B: MACHINE LEARNING FOR SEVERE EDUCATIONAL DELAY
                 metric         value
0                 n_obs  54058.000000
1  severe_offtrack_rate      0.398479
2     mean_years_behind      1.515613
3   median_years_behind      1.000000

Running ML model: Logistic Regression
Running ML model: Random Forest

                 model  best_cv_avg_precision  test_accuracy  test_roc_auc  \
1        Random Forest               0.720117       0.733074      0.807327   
0  Logistic Regression               0.696008       0.719694      0.784428   

   test_avg_precision  test_precision  test_recall   test_f1  \
1            0.727789        0.697812     0.582392  0.634899   
0            0.695280        0.667132     0.591985  0.

<Figure size 700x600 with 0 Axes>

<Figure size 700x600 with 0 Axes>